# TRIBE Studio — Colab GPU backend + **ngrok** (official CLI)

This notebook runs **`backend/`** on a **Colab GPU**, starts **uvicorn** on port **8000**, then starts the **official ngrok agent** (`ngrok http 8000`). Your Mac backend uses the printed **`REMOTE_TRIBE_URL`**.

**Security:** never commit your ngrok authtoken. If it was pasted into chat or a repo, **revoke it** in the [ngrok dashboard](https://dashboard.ngrok.com/) and create a new one.

**Do not** set `REMOTE_TRIBE_URL` on Colab (infinite loop with the Mac forwarder).

In [ ]:
# @title 0) GPU check
import torch
print("cuda:", torch.cuda.is_available(), "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a")

### 1) Put this repo on the Colab VM

- **Option A:** set `REPO_URL` to your GitHub fork below.
- **Option B:** upload `eureka-hacks.zip` (must contain `backend/`) to `/content/`, set `USE_ZIP=True`.

In [ ]:
# @title 1) Clone or unzip
import shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/eureka-hacks.git"  # <-- edit
BRANCH = "main"
ROOT = Path("/content/eureka-hacks")

if ROOT.exists():
    shutil.rmtree(ROOT)

USE_ZIP = False

if USE_ZIP:
    z = Path("/content/eureka-hacks.zip")
    if not z.is_file():
        raise FileNotFoundError("Upload eureka-hacks.zip to /content/ or set USE_ZIP=False and fix REPO_URL")
    subprocess.check_call(["unzip", "-q", str(z), "-d", str(ROOT.parent)])
else:
    if "YOUR_GITHUB_USERNAME" in REPO_URL:
        raise RuntimeError("Edit REPO_URL to your fork (or set USE_ZIP=True).")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(ROOT)])

BACKEND = ROOT / "backend"
assert (BACKEND / "main.py").is_file(), f"Missing backend/main.py under {BACKEND}"
print("OK:", BACKEND)

In [ ]:
# @title 2) Python deps (may take several minutes)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(BACKEND / "requirements.txt")])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/tribev2.git"])

In [ ]:
# @title 3) Hugging Face login (gated Llama for TRIBE)
import os, getpass

os.environ.pop("REMOTE_TRIBE_URL", None)
os.environ.pop("TRIBE_DEMO", None)

try:
    from huggingface_hub import login
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import login

token = os.environ.get("HF_TOKEN") or getpass.getpass("HF read token (hidden): ").strip()
if not token:
    raise RuntimeError("Missing HF token")
login(token=token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = token
print("HF login OK")

In [ ]:
# @title 4) Stop old uvicorn / ngrok (re-run friendly)
import subprocess, time
subprocess.run("pkill -f 'uvicorn main:app' || true", shell=True)
subprocess.run("pkill -f '[n]grok' || true", shell=True)
time.sleep(1)

In [ ]:
# @title 4b) Install official ngrok CLI (Colab = Debian/Ubuntu)
import shutil, subprocess

if shutil.which("ngrok"):
    print("ngrok already on PATH:", shutil.which("ngrok"))
else:
    print("Installing ngrok via apt (per https://ngrok.com/download Linux / apt ) …")
    install = r'''
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y curl gnupg
curl -fsSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc \
  | tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
echo "deb https://ngrok-agent.s3.amazonaws.com buster main" \
  | tee /etc/apt/sources.list.d/ngrok.list >/dev/null
apt-get update -qq
apt-get install -y ngrok
'''
    subprocess.run(["bash", "-lc", install], check=True)
    print("ngrok:", shutil.which("ngrok"))

In [ ]:
# @title 5) Register authtoken, start uvicorn + `ngrok http 8000`, print Mac export
import json, os, shutil, subprocess, sys, time, getpass
from pathlib import Path
from urllib.request import urlopen

if not shutil.which("ngrok"):
    raise RuntimeError("Run cell 4b: ngrok is not on PATH.")

os.chdir(BACKEND)
os.environ.pop("REMOTE_TRIBE_URL", None)
os.environ["TRIBE_DEVICE"] = os.environ.get("TRIBE_DEVICE", "cuda")

ng_token = getpass.getpass("ngrok authtoken (dashboard → Your Authtoken): ").strip()
subprocess.run(["ngrok", "config", "add-authtoken", ng_token], check=True)

log_path = Path("/content/uvicorn.log")
log_f = log_path.open("wb")
uv = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(BACKEND),
    stdout=log_f,
    stderr=subprocess.STDOUT,
)
time.sleep(3)

ng = subprocess.Popen(
    ["ngrok", "http", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

public_url = None
for _ in range(120):
    try:
        data = json.load(urlopen("http://127.0.0.1:4040/api/tunnels", timeout=2))
        for t in data.get("tunnels", []):
            u = (t.get("public_url") or "").strip()
            if u.startswith("https://"):
                public_url = u.rstrip("/")
                break
    except Exception:
        pass
    if public_url:
        break
    time.sleep(1)

try:
    log_f.flush()
finally:
    log_f.close()

if not public_url:
    print("No HTTPS tunnel from ngrok. Uvicorn log tail:")
    print(log_path.read_text(errors="replace")[-8000:])
    raise SystemExit(1)

analyze_url = public_url + "/api/analyze"
print("\n--- On your Mac (terminal that runs local uvicorn) ---")
print(f'export REMOTE_TRIBE_URL="{analyze_url}"')
print("unset TRIBE_DEMO")
print("# then: cd backend && source .venv/bin/activate && python -m uvicorn main:app --reload --port 8000")
print("\nTunnel base:", public_url)
print("\nTip: on this VM you can re-print the URL with:")
print("python /content/eureka-hacks/scripts/ngrok_print_analyze_url.py")

### VS Code / Cursor Colab extension

1. Install Google’s **Colab** extension.
2. Open this notebook, kernel → **Colab** → **GPU**.
3. Run cells **in order** (0 → 5). Keep the runtime alive while you use the Mac UI.

If `apt-get install ngrok` fails (permissions), use a Colab runtime where you are root, or install ngrok manually in the VM and re-run from cell 4b.